In [ ]:
from crewai import Agent,Task,Crew,Process,LLM
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient
from typing import List
from crewai.tools import tool
from scrapegraph_py import Client
import agentops
import os 
import json

In [74]:
load_dotenv()

llm = LLM(
    model="command-a-03-2025",
    api_key=os.getenv("COHERE_API_KEY"),
    base_url="https://api.cohere.com/compatibility/v1",
    temperature=0
)

In [75]:
session = agentops.init(
    api_key=os.getenv("AGENTOPS_API_KEY"),
    #this code to prevent close after first agent 
    skip_auto_end_session=True
)
print(session)

None


In [ ]:
search_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
scrape_client = Client(api_key=os.getenv("ScrapGraphAI_API_KEY"))

NameError: name 'TavilyClient' is not defined

In [77]:
outpur_dir = "./ai-agent-output"
os.makedirs(outpur_dir,exist_ok=True)

### FIRST AGENT & TASK

In [78]:
no_keywords = 10
class SuggestedSearchQueries(BaseModel):
    queries: List[str] = Field(...,title="suggested search quieries to be passed to the search engine")

#every agent can make one task or multiple tasks
#each agent has role, goal, backstory,llm  
Search_Query_Recommendation_Agent = Agent(
    role="Search Query Strategist",
    goal="/n".join([
        "to provide alist of suggested search queries to be passed to the search engine",
        "the queries must be varied and looking for specific items"
           ]),
    backstory="You are an expert search query strategist specializing in product research and online shopping. You analyze user requests and transform them into clear, specific, and diverse search queries. Your goal is to cover different brands, models, specifications, price ranges, and purchasing options to help search engines find the most relevant products and deals."    ,
    llm = llm,
    verbose=True
)
#each task has description , expected output and agent to do this task , expected output , output json
# optional can take async,output file 
Search_Query_Recommendation_Task = Task(
    description="\n".join([
        "Generate exactly {no_keyword} highly relevant and diverse search keywords for finding {product_name}.",

        "The keywords will be passed to a separate search engine agent that will perform the actual web search.",

        "The product must be available for purchase and deliverable in {country_name}.",

        "The search will be restricted to these e-commerce websites:",
        "{website_list}",

        "IMPORTANT: Do NOT include website names, domains, URLs, or 'site:' operators in the keywords.",

        "Do NOT generate URLs or links.",

        "Do NOT return product pages or search results.",

        "Do NOT include blog posts, articles, reviews, forums, news websites, or informational content.",

        "Generate keywords that describe the product and its purchasing requirements.",

        "Vary the keywords using different brands, models, specifications, features, price ranges, and product variations.",

        "Each keyword should be a concise search phrase that can be directly passed to a search engine.",

        "Return ONLY the list of keywords.",
    ]),
    #json to prevent words in introduction and conclusion
    expected_output="A JSON containing alist of suggested search queries",
    #we will create pydantic scheme for output json 
    output_json=SuggestedSearchQueries,
    output_file=os.path.join(outpur_dir,"step1.json"),
    agent=Search_Query_Recommendation_Agent
)

### SECOND AGENT

In [ ]:
#create nested pydantic
class SingleSearchResult(BaseModel):
    title: str
    url: str
    content: str
    score: float
    search_query: str
    
class AllSearchResult(BaseModel):
    result:List[SingleSearchResult]


#  we need google search tool
#will make custom tool (tavily)
# doc string is must in custom tool to tell what tool make
@tool
def search_engine_tool(query:str):
    """
    Search the web for relevant e-commerce product pages using the provided query.

    Args:
        query: A search query describing the product or information to find.
                The query should be specific and focused on purchasable products.

    Returns:
        Search results containing relevant web pages matching the query.
    """
    return  search_client.search(
    query=query,
    search_depth="advanced",
    max_results=5
)


# take keywords and search google for products based on the suggested search query
Search_Engine_Agent = Agent(
    role = "Search Engine Agent",
    goal = "To search for products based on the suggessted search query",
    backstory="""
    You are an expert web search specialist focused on finding
    purchasable products across e-commerce websites.

    You receive carefully generated search queries from a Search Query
    Recommendation Agent and use them to perform accurate web searches.

    Your responsibility is to find relevant product pages, product listings,
    and online stores. You prioritize results that contain actual products
    available for purchase and ignore blogs, news articles, forums, and
    informational pages.

    You carefully follow the provided search queries and return the most
    relevant search results without modifying the user's search intent.
    """,
    llm = llm,
    verbose=True,
    tools=[search_engine_tool]

)

Search_Engine_Task = Task(
    description="/n".join([
        "The task is to search for products based on the suggested search queries.",
        "You have to collect results from multiple search queries.",
        "Ignore any susbicious links or not an ecommerce single product website link.",
        "Ignore any search results with confidence score less than ({score_th}) .",
        "The search results will be used to compare prices of products from different websites."
    ]),
    expected_output="A JSON object containing the search results",
    output_json=AllSearchResult,
    output_file=os.path.join(outpur_dir,"step2.json"),
    context=[Search_Query_Recommendation_Task],
    agent=Search_Engine_Agent
)

### third Agent

In [ ]:
class ProductSpec(BaseModel):
    specification_name: str
    specification_value: str

class SingleExtractedProduct(BaseModel):
    page_url: str = Field(..., title="The original url of the product page")
    product_title: str = Field(..., title="The title of the product")
    product_image_url: str = Field(..., title="The url of the product image")
    product_url: str = Field(..., title="The url of the product")
    product_current_price: float = Field(..., title="The current price of the product")
    product_original_price: float = Field(title="The original price of the product before discount. Set to None if no discount", default=None)
    product_discount_percentage: float = Field(title="The discount percentage of the product. Set to None if no discount", default=None)

    product_specs: List[ProductSpec] = Field(..., title="The specifications of the product. Focus on the most important specs to compare.", min_items=1, max_items=5)

    agent_recommendation_rank: int = Field(..., title="The rank of the product to be considered in the final procurement report. (out of 5, Higher is Better) in the recommendation list ordering from the best to the worst")
    agent_recommendation_notes: List[str]  = Field(..., title="A set of notes why would you recommend or not recommend this product to the company, compared to other products.")


class AllExtractedProducts(BaseModel):
    products: List[SingleExtractedProduct]


#we need custom tool to descripe , extract data from webpages (ScrapGraphAi)
@tool
def web_scraping_tool(page_url: str,required_fields: list):
    # doc string is must in custom tool to tell what tool make
    """
    An AI Tool to help an agent to scrape a web page

    Example:
    web_scraping_tool(
        page_url="https://www.noon.com/egypt-en/15-bar-fully-automatic-espresso-machine-1-8-l-1500"
    )
    """
    details = scrape_client.smartscraper(
        website_url=page_url,
        user_prompt="Extract ```json\n" + SingleExtractedProduct.schema_json()+ "```\nFrom the web page"
    )
    return {
        "page_url":page_url,
        "details":details
    }


# take web result and download page  to extract details from any web page
Scraping_Agent = Agent(
    role = "web scraping agent",
    goal = "to extract details from any website",
    backstory="The agent is designed to help in looking for required values from any website url. These details will be used to decide which best product to buy.",
    llm =llm ,
    tools = [web_scraping_tool],
    verbose = True
)
Scraping_Task = Task(
    description="\n".join([
        "The task is to extract product details from any ecommerce store page url.",
        "The task has to collect results from multiple pages urls.",
        "Collect the best {top_recommendations_no} products from the search results.",       
    ])
    expected_output="A JSON object containing products details",
    output_json=AllExtractedProducts,
    output_file=(os.path.join(outpur_dir,"step3.json")),
    agent=Scraping_Agent
)

In [80]:
run = Crew(
    agents=[
        Search_Query_Recommendation_Agent,
        Search_Engine_Agent, 
            ],
    tasks=[
        Search_Query_Recommendation_Task,
        Search_Engine_Task
        ],
    process=Process.sequential
)

In [81]:
results =  await run.kickoff_async(
    inputs={
        "product_name": "coffee machine for the office",
        "website_list": ["www.amazon.eg", "www.jumia.com.eg", "www.noon.com/egypt-en"],
        "country_name": "Egypt",
        "no_keyword": 10,
        "score_th":0.6
    }
    )

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Search Query Strategist                                                                                 │
│                                                                                                                 │
│  Task: Generate exactly 10 highly relevant and diverse search keywords for finding coffee machine for the       │
│  office.                                                                                                        │
│  The keywords will be passed to a separate search engine agent that will perform the actual web search.         │
│  The product must be available for purchase and deliverable in Egypt.                                           │
│  The search will be restricted to these e-commerce websites:                                                    │
│  ['www.amazon.eg', 'www.jumia.com.eg', 'www.noon.com/egypt-en']                                                 │
│  IMPORTANT: Do NOT include website names, domains, URLs, or 'site:' operators in the keywords.                  │
│  Do NOT generate URLs or links.                                                                                 │
│  Do NOT return product pages or search results.                                                                 │
│  Do NOT include blog posts, articles, reviews, forums, news websites, or informational content.                 │
│  Generate keywords that describe the product and its purchasing requirements.                                   │
│  Vary the keywords using different brands, models, specifications, features, price ranges, and product          │
│  variations.                                                                                                    │
│  Each keyword should be a concise search phrase that can be directly passed to a search engine.                 │
│  Return ONLY the list of keywords.                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Search Query Strategist                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  queries=['Office coffee machines Egypt', 'Commercial coffee makers for offices in Egypt', 'Automatic espresso  │
│  machines for office use Egypt', 'Best coffee machines for large offices Egypt', 'Affordable office coffee      │
│  machines in Egypt', 'Nespresso professional coffee machines Egypt', 'Delonghi office coffee machines Egypt',   │
│  'Bean-to-cup coffee machines for offices Egypt', 'High-capacity coffee machines for offices Egypt', 'Office    │
│  coffee machines with milk frother Egypt']                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Search Engine Agent                                                                                     │
│                                                                                                                 │
│  Task: The task is to search for products based on the suggested search queries./nYou have to collect results   │
│  from multiple search queries./nIgnore any susbicious links or not an ecommerce single product website          │
│  link./nIgnore any search results with confidence score less than (0.6) ./nThe search results will be used to   │
│  compare prices of products from different websites.                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_engine_tool executed with result: {'query': 'Office coffee machines Egypt', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.fajtradingllc.com/pages/coffee-machines-in-egypt', 'title': 'Buy t...
Tool search_engine_tool executed with result: {'query': 'Commercial coffee makers for offices in Egypt', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://ecoffee1.com/best-coffee-machines-egypt-cafes-offices...
Tool search_engine_tool executed with result: {'query': 'Automatic espresso machines for office use Egypt', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://ecoffee1.com/best-coffee-machines-egypt-cafes-offi...
Tool search_engine_tool executed with result: {'query': 'Best coffee machines for large offices Egypt', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.fajtradingllc.com/pages/coffee-machines-in-egypt',...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Search Engine Agent                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│      "result": [                                                                                                │
│          {                                                                                                      │
│              "title": "Buy the Best Coffee Machines in Egypt - F A J Trading L.L.C",                            │
│              "url": "https://www.fajtradingllc.com/pages/coffee-machines-in-egypt",                             │
│              "content": "F A J Trading L.L.C is the best dealer and supplier of coffee machines designed to     │
│  help you brew the perfect cup at home, in the office, or at a restaurant. Whether you prefer rich espresso,    │
│  frothy cappuccino, or traditional Arabic coffee, you’ll find exactly what you need in Cairo, Elexendra, Suez,  │
│  Demitta, Sharkia, Port said, Kaliubia, Giza, Dakhalia, Shakira, Qaliubia, Ismailia, Borg el Arab, Menufia,     │
│  Behaira, Beni Suef Faiyum, Minya in Egypt.We offer a variety of machines, including Nespresso for              │
│  high-pressure extraction, as well as brands like De’Longhi, La Marzocco, Dr. Coffee, ECM, Conti, Faema,        │
│  Slayer, Coffee Blender, Marco Boiler, and Baratza grinders. With our extensive range, you can enjoy your       │
│  favourite coffee just as you like it. [...] + All Navigation\n + - Coffee Machines Navigation\n -              │
│  Professional Coffee Machines\n - Automatic Coffee Machine\n - Office Coffee Machines\n - Home Coffee           │
│  Machines\n - Capsule Coffee Machines\n - Coffee Brewers\n - Turkish Coffee Machines\n - Coffee Machine         │
│  Equipment\n - Coffee Accessories\n + - Home Appliances Navigation\n - Refrigerators\n - Washing Machines\n -   │
│  Cooker\n - Ovens\n - Dishwashers\n - Kitchen Hoods\n - Vacuum Cleaners\n - Hobs\n - Small Appliances\n + -     │
│  Kitchen Appliances Navigation\n - Microwaves\n - Mixer Grinder\n - BBQ Grill\n + - Water Heater Navigation\n   │
│  - Solar Water Heater\n - Electric Water Heater\n - Under Sink Water Heater Drivers\n + - Spare Parts &         │
│  Accessories Navigation\n - AC Spare Parts Navigation [...] ## Bean to Cup Coffee & Espresso                    │
│  Machines\n\nUnlock professional performance at home with Sage. Like a professional bean to cup coffee          │
│  machine, ours uses the 4 Keys Formula to deliver delicious third-wave speciality coffee.\n\nShop Now\n\nF A J  │
│  Trading L.L.C\n\n### Boost productivity with cafe-quality office coffee.\n\nThe Commercial Espresso Coffee     │
│  Machine Jetinno JL32 Espresso Coffee Machine is a top-tier bean-to-cup machine.\n\nShop now\n\nF A J Trading   │
│  L.L.C\n\n### Deliver a Premium Coffee Experience to Your Commercial Space\n\nThe Synchronika II builds on      │
│  ECM’s legacy by adding exciting features like an OLED PID display and cartridge heaters in the group           │
│  head.\n\nShop Now\n\n## Which machine works for you?\n\nDe'Longhi Dinamica ECAM350.55.B Fully Automatic        │
│  Coffee Machine – Black\n\nDe'Longhi,                                                                           │
│              "score": 0.8889231,                                                                                │
│              "search_query": "Office coffee machines Eg

CancelledError: 

In [ ]:
results

CrewOutput(raw='{"queries":["Office coffee machines Egypt delivery","Commercial coffee makers for offices in Egypt","Automatic espresso machines for office use Egypt","Best coffee machines for large offices Egypt","Affordable office coffee machines Egypt","Nespresso professional coffee machines Egypt","Bean-to-cup coffee machines for offices Egypt","High-capacity coffee makers for offices Egypt","Office coffee machines with milk frother Egypt","Energy-efficient office coffee machines Egypt"]}', pydantic=None, json_dict={'queries': ['Office coffee machines Egypt delivery', 'Commercial coffee makers for offices in Egypt', 'Automatic espresso machines for office use Egypt', 'Best coffee machines for large offices Egypt', 'Affordable office coffee machines Egypt', 'Nespresso professional coffee machines Egypt', 'Bean-to-cup coffee machines for offices Egypt', 'High-capacity coffee makers for offices Egypt', 'Office coffee machines with milk frother Egypt', 'Energy-efficient office coffee m